# Eval — Resume from baseline (Kaggle)

Notebook tiếp tục eval sau khi đã có `__base___tier{A,B1,B2,B3,C}.json`.

**Workflow:**
1. Load 5 file `__base__` JSON đã có → đưa vào summary
2. Auto-detect adapter còn lại trong `qlora_runs/` chưa eval
3. Chạy benchmark 5-tier cho từng adapter còn thiếu
4. Apply selection rule (ACCEPT/REJECT vs baseline)
5. Backup

## Pre-flight (Kaggle)
1. Settings → Accelerator: **GPU T4 / P100**
2. Settings → Internet: **ON**
3. Settings → Add Input:
   - Output cũ chứa `eval/` (5 file `__base___*.json`) + `qlora_runs/` (adapters)
   - Domain dataset (`question_bank.jsonl` + `units.jsonl`)
4. Run All.

## 0. Setup

In [ ]:
%%capture
!pip install -q --no-deps bitsandbytes accelerate peft "trl<0.12.0"
!pip install -q -U unsloth
!pip install -q -U datasets sentencepiece einops qwen-vl-utils
!pip install -q rouge-score bert-score evaluate

In [ ]:
import os, json, gc, random, re, glob, shutil
from pathlib import Path
from collections import defaultdict
from typing import Optional, Dict, Any, List

import torch
import numpy as np
from datasets import load_dataset, Dataset
from unsloth import FastVisionModel, is_bfloat16_supported

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

BASE_MODEL = "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit"
EVAL_DIR = Path("/kaggle/working/eval"); EVAL_DIR.mkdir(parents=True, exist_ok=True)
LETTERS = list("ABCDEFGHIJ")
SYSTEM = ("You are an AI/ML/NLP/CV tutor. Explain concepts clearly, accurately, "
          "and step by step in English.")

print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("bf16:", is_bfloat16_supported())

## 1. Locate inputs + import 5 file __base__

**Tự động:**
- Tìm `qlora_runs/` (chứa adapters) trong `/kaggle/input/**/` và `/kaggle/working/`
- Tìm `__base___tier*.json` trong `/kaggle/input/**/` → copy vào `EVAL_DIR`
- Tìm `question_bank.jsonl` + `units.jsonl`

In [ ]:
def _find_dir(name):
    for root in ["/kaggle/working", "/kaggle/input"]:
        for p in glob.glob(f"{root}/**/{name}", recursive=True):
            if Path(p).is_dir(): return Path(p)
    return None

def _find_file(name):
    for root in ["/kaggle/input", "/kaggle/working"]:
        for p in glob.glob(f"{root}/**/{name}", recursive=True):
            return Path(p)
    return None

# Auto-unzip nếu user attach zip
for zip_p in glob.glob("/kaggle/input/**/qlora_runs*.zip", recursive=True):
    target = Path("/kaggle/working/qlora_runs_unzipped")
    if not target.exists():
        print(f"Unzip {zip_p} -> {target}")
        shutil.unpack_archive(zip_p, target)

RUNS_DIR   = _find_dir("qlora_runs")
QB_PATH    = _find_file("question_bank.jsonl")
UNITS_PATH = _find_file("units.jsonl")

# Import 5 file __base___tier*.json từ input dataset vào EVAL_DIR
imported = 0
for tier in ["tierA", "tierB1", "tierB2", "tierB3", "tierC"]:
    src = _find_file(f"__base___{tier}.json")
    if src is None: continue
    dst = EVAL_DIR / src.name
    if not dst.exists():
        shutil.copy(src, dst); imported += 1
    print(f"  {tier}: {src} -> {dst}")

# Tier A details (optional)
src_d = _find_file("__base___tierA_details.jsonl")
if src_d:
    dst_d = EVAL_DIR / src_d.name
    if not dst_d.exists(): shutil.copy(src_d, dst_d)

print(f"\n  Imported {imported} baseline files")
print("Adapters root :", RUNS_DIR)
print("QB            :", QB_PATH)
print("Units         :", UNITS_PATH)

assert RUNS_DIR is not None, "Không tìm thấy qlora_runs/. Attach Kaggle Output cũ."
assert QB_PATH and UNITS_PATH, "Thiếu domain data."

RUNS = sorted([p.name for p in RUNS_DIR.iterdir() if (p/"adapter").exists()])
ALL_NAMES = ["__base__"] + RUNS
print("Adapters:", RUNS)

# Verify baseline đầy đủ
for tier in ["tierA","tierB1","tierB2","tierB3","tierC"]:
    p = EVAL_DIR / f"__base___{tier}.json"
    print(f"  __base__ {tier}: {'OK' if p.exists() else 'MISSING'}")

## 2. Domain test set (cùng SEED=42 với khi train)

In [ ]:
def mcq_to_chat(item):
    q = item["question"].strip()
    choices = item["choices"]; idx = item["answer_index"]
    expl = (item.get("explanation") or "").strip()
    if idx is None or idx >= len(choices): return []
    correct = LETTERS[idx]
    choice_text = "\n".join(f"({LETTERS[i]}) {c}" for i,c in enumerate(choices))
    user1 = f"{q}\n\n{choice_text}\n\nWhich option is correct?"
    asst1 = f"The correct answer is ({correct}). {expl}".strip()
    base = {"lecture_id": item.get("lecture_id","unknown"),
            "course_id": item.get("course_id",""), "src":"qb"}
    return [{"user":user1,"assistant":asst1,**base,"variant":"mcq"},
            {"user":q,"assistant":expl or choices[idx],**base,"variant":"open"}]

def unit_to_chat(item):
    title = item.get("lecture_title") or item.get("lecture_id","")
    name = item.get("unit_name") or ""
    summary = (item.get("summary") or "").strip()
    if not summary: return None
    kps = item.get("key_points") or []
    bullets = "\n".join(f"- {kp['text']}" for kp in kps if kp.get("text"))
    return {"user":f"Explain this lecture topic: {title} — {name}",
            "assistant":summary+(("\n\nKey points:\n"+bullets) if bullets else ""),
            "lecture_id":item.get("lecture_id","unknown"),
            "course_id":item.get("course_id",""),"src":"units","variant":"summary"}

rows = []
for line in open(QB_PATH, encoding="utf-8"):
    try: o = json.loads(line)
    except: continue
    if not o.get("qa_gate_passed", True): continue
    rows.extend(mcq_to_chat(o))
for line in open(UNITS_PATH, encoding="utf-8"):
    try: o = json.loads(line)
    except: continue
    if not o.get("active", True): continue
    r = unit_to_chat(o)
    if r: rows.append(r)
DOMAIN_DS = Dataset.from_list(rows)

def split_by_lecture(ds, val_ratio=0.10, test_ratio=0.10):
    lectures = sorted({r["lecture_id"] for r in ds if r["lecture_id"] != "eli5"})
    rng = random.Random(SEED); rng.shuffle(lectures)
    n = len(lectures); n_test = max(1,int(n*test_ratio)); n_val = max(1,int(n*val_ratio))
    test_l = set(lectures[:n_test]); val_l = set(lectures[n_test:n_test+n_val])
    by = defaultdict(list)
    for r in ds:
        b = "test" if r["lecture_id"] in test_l else ("val" if r["lecture_id"] in val_l else "train")
        by[b].append(r)
    return Dataset.from_list(by["train"]), Dataset.from_list(by["val"]), Dataset.from_list(by["test"])

_, _, DOMAIN_TEST = split_by_lecture(DOMAIN_DS)
print("Test MCQ items:", sum(1 for r in DOMAIN_TEST if r['variant']=='mcq'))

## 3. Inference helpers (đã fix processor bug)

In [ ]:
ANS_RX = re.compile(r"\b(?:answer\s+is|correct\s+answer\s+is|answer:)\s*\(?([A-J])\)?", re.I)
ANS_FALLBACK = re.compile(r"^\s*\(?([A-J])\)", re.I)

def parse_letter(text):
    m = ANS_RX.search(text) or ANS_FALLBACK.search(text.strip())
    return m.group(1).upper() if m else None

def load_adapter(run_name, max_seq_len=2048):
    path = BASE_MODEL if run_name == "__base__" else str(RUNS_DIR/run_name/"adapter")
    model, tok = FastVisionModel.from_pretrained(path, load_in_4bit=True, max_seq_length=max_seq_len)
    FastVisionModel.for_inference(model)
    return model, tok

@torch.no_grad()
def generate(model, processor, q, max_new_tokens=400):
    msgs = [
        {"role":"system","content":[{"type":"text","text":SYSTEM}]},
        {"role":"user",  "content":[{"type":"text","text":q}]},
    ]
    prompt = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = processor(text=[prompt], images=None, videos=None,
                    return_tensors="pt", padding=True).to(model.device)
    out = model.generate(**inp, max_new_tokens=max_new_tokens, max_length=None,
                         do_sample=False, temperature=0.0, repetition_penalty=1.05)
    gen = out[0][inp["input_ids"].shape[1]:]
    return processor.batch_decode([gen], skip_special_tokens=True)[0]

## 4. Benchmark functions (Tier A/B1/B2/B3/C)

In [ ]:
MMLU_SUBJECTS = ["machine_learning","college_computer_science",
                 "college_mathematics","high_school_statistics"]

def benchmark_internal_mcq(run_name, max_items=200):
    mcq_test = [r for r in DOMAIN_TEST if r["variant"] == "mcq"][:max_items]
    if not mcq_test: return None
    model, tok = load_adapter(run_name)
    correct = 0; per_course = defaultdict(lambda:[0,0]); rows = []
    for r in mcq_test:
        out = generate(model, tok, r["user"], max_new_tokens=200)
        pred = parse_letter(out)
        gold = r["assistant"].split("(",1)[1][0] if "(" in r["assistant"] else None
        ok = (pred == gold); correct += int(ok)
        per_course[r["course_id"]][0] += int(ok); per_course[r["course_id"]][1] += 1
        rows.append({"q":r["user"][:200],"pred":pred,"gold":gold,"ok":ok,"course":r["course_id"]})
    res = {"run":run_name,"tier":"A_internal_mcq","n":len(mcq_test),
           "accuracy":round(correct/len(mcq_test),4),
           "by_course":{c:round(v[0]/v[1],3) for c,v in per_course.items()}}
    (EVAL_DIR/f"{run_name}_tierA.json").write_text(json.dumps(res,indent=2))
    with open(EVAL_DIR/f"{run_name}_tierA_details.jsonl","w",encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r,ensure_ascii=False)+"\n")
    del model; gc.collect(); torch.cuda.empty_cache()
    return res

def benchmark_mmlu(run_name, max_per_subject=50):
    model, tok = load_adapter(run_name); results = {}
    for subj in MMLU_SUBJECTS:
        try: ds = load_dataset("cais/mmlu", subj, split="test")
        except Exception as e: print(f"[skip {subj}]: {e}"); continue
        if len(ds) > max_per_subject: ds = ds.shuffle(seed=SEED).select(range(max_per_subject))
        ok = 0
        for ex in ds:
            choices = "\n".join(f"({LETTERS[i]}) {c}" for i,c in enumerate(ex["choices"]))
            prompt = f"{ex['question']}\n\n{choices}\n\nWhich option is correct?"
            pred = parse_letter(generate(model, tok, prompt, max_new_tokens=50))
            ok += int(pred == LETTERS[ex["answer"]])
        results[subj] = {"n":len(ds),"acc":round(ok/len(ds),4)}
    res = {"run":run_name,"tier":"B1_mmlu","subjects":results,
           "avg_acc":round(np.mean([r["acc"] for r in results.values()]),4) if results else 0.0}
    (EVAL_DIR/f"{run_name}_tierB1.json").write_text(json.dumps(res,indent=2))
    del model; gc.collect(); torch.cuda.empty_cache()
    return res

def benchmark_mmlu_pro(run_name, max_items=100):
    model, tok = load_adapter(run_name)
    try: ds = load_dataset("TIGER-Lab/MMLU-Pro", split="test")
    except Exception as e:
        print(f"[mmlu-pro skip]: {e}"); del model; gc.collect(); torch.cuda.empty_cache(); return None
    keep = {"computer science","math","engineering","physics"}
    ds = ds.filter(lambda x: x.get("category","").lower() in keep)
    if len(ds) > max_items: ds = ds.shuffle(seed=SEED).select(range(max_items))
    ok = 0
    for ex in ds:
        opts = ex["options"]
        choices = "\n".join(f"({LETTERS[i]}) {c}" for i,c in enumerate(opts))
        prompt = f"{ex['question']}\n\n{choices}\n\nWhich option is correct?"
        pred = parse_letter(generate(model, tok, prompt, max_new_tokens=80))
        ok += int(pred == ex["answer"])
    res = {"run":run_name,"tier":"B2_mmlu_pro","n":len(ds),
           "accuracy":round(ok/len(ds),4) if len(ds) else 0.0}
    (EVAL_DIR/f"{run_name}_tierB2.json").write_text(json.dumps(res,indent=2))
    del model; gc.collect(); torch.cuda.empty_cache()
    return res

def benchmark_theoremqa(run_name, max_items=100):
    model, tok = load_adapter(run_name)
    try: ds = load_dataset("TIGER-Lab/TheoremQA", split="test")
    except Exception as e:
        print(f"[theoremqa skip]: {e}"); del model; gc.collect(); torch.cuda.empty_cache(); return None
    if len(ds) > max_items: ds = ds.shuffle(seed=SEED).select(range(max_items))
    correct = 0
    for ex in ds:
        pred = generate(model, tok, ex["Question"], max_new_tokens=200).strip().lower()
        gold = str(ex.get("Answer","")).strip().lower()
        if gold and gold in pred: correct += 1
    res = {"run":run_name,"tier":"B3_theoremqa","n":len(ds),
           "loose_match_acc":round(correct/len(ds),4)}
    (EVAL_DIR/f"{run_name}_tierB3.json").write_text(json.dumps(res,indent=2))
    del model; gc.collect(); torch.cuda.empty_cache()
    return res

QTYPE_RX = re.compile(r"^\s*(why|how|what happens|how does|how do|what is the difference|difference between)\b", re.I)
TOPIC_KW = ["machine learning","deep learning","neural network","transformer","embedding",
    "language model","computer vision","cnn","convolution","probability","statistic",
    "optimization","gradient","algorithm","science","math","computer","data"]
EXCLUDE_KW = ["trump","election","politic","celebrity","basketball","nba","nfl","movie"]
def _has_any(t,k): return any(x in t.lower() for x in k)
def _eli5_keep(q,a):
    if not (q and a): return False
    if not QTYPE_RX.search(q): return False
    wc = len(a.split())
    if wc < 120 or wc > 450: return False
    if _has_any(q+" "+a, EXCLUDE_KW): return False
    return _has_any(q+" "+a, TOPIC_KW)

def load_eli5_dev(n=50):
    raw = load_dataset("sentence-transformers/eli5", split="train")
    rows = []
    for ex in raw:
        q = ex.get("question") or ""; a = ex.get("answer") or ""
        if _eli5_keep(q,a): rows.append({"user":q,"assistant":a})
        if len(rows) >= n*3: break
    ds = Dataset.from_list(rows)
    return ds.shuffle(seed=SEED).select(range(min(n,len(ds))))

def benchmark_eli5_style(run_name, max_items=50):
    from rouge_score import rouge_scorer
    from bert_score import score as bert_score
    eli5_dev = load_eli5_dev(max_items)
    model, tok = load_adapter(run_name)
    preds, refs, lens = [], [], []
    for r in eli5_dev:
        out = generate(model, tok, r["user"], max_new_tokens=400)
        preds.append(out); refs.append(r["assistant"]); lens.append(len(out.split()))
    del model; gc.collect(); torch.cuda.empty_cache()
    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge_l = float(np.mean([rouge.score(r,p)["rougeL"].fmeasure for p,r in zip(preds,refs)]))
    _,_,F = bert_score(preds, refs, lang="en", verbose=False)
    res = {"run":run_name,"tier":"C_eli5_style","n":len(preds),
           "rouge_l_f1":round(rouge_l,4),
           "bertscore_f1":round(float(F.mean()),4),
           "avg_words":round(float(np.mean(lens)),1)}
    (EVAL_DIR/f"{run_name}_tierC.json").write_text(json.dumps(res,indent=2))
    return res

## 5. Run benchmark cho adapter còn thiếu (skip-aware)

**Logic:** mỗi tier check file JSON tồn tại → skip. Nếu adapter X đã có đủ 5 file → skip toàn bộ. Cho phép rerun an toàn.

In [ ]:
TIERS = [
    ("tierA",  benchmark_internal_mcq),
    ("tierB1", benchmark_mmlu),
    ("tierB2", benchmark_mmlu_pro),
    ("tierB3", benchmark_theoremqa),
    ("tierC",  benchmark_eli5_style),
]

def run_skip_aware(run_name):
    print(f"\n##### {run_name} #####")
    for tier_name, fn in TIERS:
        out = EVAL_DIR / f"{run_name}_{tier_name}.json"
        if out.exists():
            print(f"  [skip] {tier_name} (đã có)")
            continue
        print(f"  [run]  {tier_name}")
        try: fn(run_name)
        except Exception as e:
            print(f"  [FAIL] {tier_name}: {e}")

for name in ALL_NAMES:
    run_skip_aware(name)

print("\nDONE")

## 6. Summary + Selection rule

In [ ]:
import pandas as pd

def summarize():
    rows = []
    for name in ALL_NAMES:
        r = {"run": name}
        for tier, key, dst in [
            ("tierA", "accuracy",        "tierA_acc"),
            ("tierB1","avg_acc",         "tierB1_acc"),
            ("tierB2","accuracy",        "tierB2_acc"),
            ("tierB3","loose_match_acc", "tierB3_acc"),
            ("tierC", "bertscore_f1",    "tierC_bertscore"),
            ("tierC", "rouge_l_f1",      "tierC_rouge"),
            ("tierC", "avg_words",       "tierC_avg_words"),
        ]:
            p = EVAL_DIR / f"{name}_{tier}.json"
            if p.exists(): r[dst] = json.loads(p.read_text()).get(key)
        rows.append(r)
    df = pd.DataFrame(rows); df.to_csv(EVAL_DIR/"summary.csv", index=False)
    return df

df = summarize()
df

In [ ]:
def apply_selection_rule(df):
    """PIPELINE §9: ACCEPT iff tierA > base+3pt AND mmlu_drop ≤ 2pt AND bertscore ≥ baseline."""
    base_rows = df[df["run"] == "__base__"]
    if len(base_rows) == 0:
        print("Không có baseline."); return df
    base = base_rows.iloc[0].to_dict()
    verdicts = []
    for _, r in df.iterrows():
        if r["run"] == "__base__": verdicts.append("BASELINE"); continue
        reasons = []
        if r.get("tierA_acc",0) < (base.get("tierA_acc",0) or 0) + 0.03:
            reasons.append("tierA improvement < 3pt")
        if r.get("tierB1_acc",0) < (base.get("tierB1_acc",0) or 0) - 0.02:
            reasons.append("MMLU drop > 2pt")
        if r.get("tierC_bertscore",0) < (base.get("tierC_bertscore",0) or 0):
            reasons.append("BERTScore regression")
        verdicts.append("ACCEPT" if not reasons else f"REJECT: {'; '.join(reasons)}")
    df = df.copy(); df["verdict"] = verdicts
    return df

df_v = apply_selection_rule(df)
df_v.to_csv(EVAL_DIR/"summary_with_verdict.csv", index=False)
df_v

## 7. Backup

In [ ]:
shutil.make_archive("/kaggle/working/eval_full", "zip", "/kaggle/working/eval")
print("Eval zip:", Path("/kaggle/working/eval_full.zip").stat().st_size/1e6, "MB")
print("Save Version để giữ output qua sessions.")